In [ ]:
# Install required packages
!pip install opencv-python-headless matplotlib scikit-learn keras-tuner -q

import os
import cv2
import shutil
import random
import numpy as np
import time
from glob import glob
from matplotlib import pyplot as plt
from google.colab import drive

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from keras_tuner import HyperModel
from keras_tuner.tuners import RandomSearch
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Mount Google Drive
drive.mount('/content/drive')

# ================== CONFIG ==================
INPUT_DIR = "/content/PRASNABUBBLES"
CROPPED_DIR = "/content/drive/MyDrive/omr_dataset/cropped_bubbles/all"
SOURCE_DIR = "/content/omr_dataset"
DEST_DIR = "/content/drive/MyDrive/omr_dataset"
CLASSES = ["filled", "empty", "invalid"]
SPLIT_RATIOS = (0.7, 0.15, 0.15)
IMG_SIZE = (64, 64)
BATCH_SIZE = 32

os.makedirs(CROPPED_DIR, exist_ok=True)

# ================== STAGE 1: BUBBLE EXTRACTION ==================

def correct_skew(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, bin_img = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    coords = np.column_stack(np.where(bin_img > 0))
    if coords.shape[0] < 10:
        return img
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle += 90
    (h, w) = img.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)

def preprocess_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Unable to load image: {image_path}")
        return None, None
    img = correct_skew(img)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY_INV, 25, 15)
    return img, thresh

def find_bubbles(thresh_img):
    contours, _ = cv2.findContours(thresh_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bubbles = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        aspect = w / float(h)
        area = cv2.contourArea(cnt)
        if 400 < area < 2500 and 20 < w < 60 and 20 < h < 60 and 0.8 < aspect < 1.2:
            bubbles.append((x, y, w, h))
    return sorted(bubbles, key=lambda b: (b[1] // 50, b[0]))

def extract_and_save_bubbles(image_path):
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    img, thresh = preprocess_image(image_path)
    if img is None:
        return
    bubbles = find_bubbles(thresh)
    if not bubbles:
        print(f"No bubbles found in: {image_path}")
        return
    for i, (x, y, w, h) in enumerate(bubbles):
        resized = cv2.resize(img[y:y+h, x:x+w], (64, 64))
        cv2.imwrite(os.path.join(CROPPED_DIR, f"{base_name}_bubble{i}.png"), resized)

all_images = [os.path.join(INPUT_DIR, f) for f in os.listdir(INPUT_DIR)
              if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
for img_path in all_images:
    extract_and_save_bubbles(img_path)
print("✅ All bubbles extracted. Ready for manual labeling.")

# ================== STAGE 2: DATASET SPLITTING ==================

for split in ["train", "val", "test"]:
    for cls in CLASSES:
        os.makedirs(os.path.join(DEST_DIR, split, cls), exist_ok=True)

for cls in CLASSES:
    files = glob(os.path.join(SOURCE_DIR, cls, "*.png"))
    random.shuffle(files)
    total = len(files)
    train_end = int(SPLIT_RATIOS[0] * total)
    val_end = train_end + int(SPLIT_RATIOS[1] * total)
    splits = {"train": files[:train_end], "val": files[train_end:val_end], "test": files[val_end:]}
    for split_name, split_files in splits.items():
        for file_path in split_files:
            shutil.copy(file_path, os.path.join(DEST_DIR, split_name, cls, os.path.basename(file_path)))

print("✅ Dataset split into train/val/test successfully.")

# ================== STAGE 3: CNN TRAINING ==================

TRAIN_DIR = os.path.join(DEST_DIR, "train")
VAL_DIR = os.path.join(DEST_DIR, "val")
TEST_DIR = os.path.join(DEST_DIR, "test")

train_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, color_mode='grayscale',
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=True)

val_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, color_mode='grayscale',
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

class CNNHyperModel(HyperModel):
    def build(self, hp):
        model = tf.keras.Sequential([
            tf.keras.Input(shape=(*IMG_SIZE, 1)),
            layers.Conv2D(hp.Choice('conv1_filters', [32, 64]), (3, 3), activation='relu'),
            layers.Conv2D(hp.Choice('conv2_filters', [32, 64]), (3, 3), activation='relu'),
            layers.MaxPooling2D(2, 2),
            layers.Conv2D(hp.Choice('conv3_filters', [64, 128]), (3, 3), activation='relu'),
            layers.Conv2D(hp.Choice('conv4_filters', [64, 128]), (3, 3), activation='relu'),
            layers.MaxPooling2D(2, 2),
            layers.Flatten(),
            layers.Dense(hp.Choice('dense_units', [64, 128, 256]), activation='relu'),
            layers.Dropout(hp.Float('dropout', 0.2, 0.5, step=0.1)),
            layers.Dense(3, activation='softmax')
        ])
        model.compile(
            optimizer=tf.keras.optimizers.Adam(hp.Float('lr', 1e-4, 1e-2, sampling='log')),
            loss='categorical_crossentropy', metrics=['accuracy'])
        return model

tuner = RandomSearch(
    CNNHyperModel(), objective='val_accuracy', max_trials=5,
    executions_per_trial=1, directory='cnn_tune',
    project_name='cnn_hyper_tuning', overwrite=True)

early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

print("Starting hyperparameter search...")
tuner.search(train_gen, validation_data=val_gen, epochs=10, callbacks=[early_stop], verbose=1)
tuner.results_summary()

best_hp = tuner.get_best_hyperparameters(1)[0]
model = CNNHyperModel().build(best_hp)

print("Training best model...")
history = model.fit(train_gen, validation_data=val_gen, epochs=20, callbacks=[early_stop], verbose=1)

model_path = "/content/drive/MyDrive/best_cnn_model.h5"
model.save(model_path)
print("✅ Model saved to:", model_path)

# ================== STAGE 4: EVALUATION ==================

test_gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, color_mode='grayscale',
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)

model = load_model(model_path)

start = time.time()
steps = int(np.ceil(test_gen.samples / test_gen.batch_size))
y_pred_probs = model.predict(test_gen, steps=steps, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes
print(f"Prediction time: {time.time() - start:.2f} seconds")

labels = list(test_gen.class_indices.keys())
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=labels))